# Interactive Cosmograph visualization of SSNs for MGnify proteins

<picture>
  <!-- Shown when the system/browser is in light mode -->
  <source srcset="notebook_resources/mgnify_wordmark_dark_on_light.svg" media="(prefers-color-scheme: light)">
  
  <!-- Fallback/Default shown in dark mode -->
  <img src="notebook_resources/mgnify_wordmark_light_on_dark.svg" width="300" alt="MGnify logo">
</picture>

This notebook serves as the interactive frontend for the **MiSSN** pipeline, allowing you to explore and visualise Sequence Similarity Networks using Cosmograph.

Before running this notebook, ensure you have the required dependencies installed and access to the necessary parquet files for the analysis. If not, please run the main pipeline first.

## Import libraries

In [ ]:
# %pip install ipyfilechooser pandas matplotlib numpy ipywidgets cosmograph duckdb

from cosmograph import cosmo
import duckdb
from ipyfilechooser import FileChooser
from IPython.display import display, clear_output
import ipywidgets as widgets
import pandas as pd

## Load data
Choose one of the options below

### Option A: Read files using remote FTP

In [ ]:
cluster_rep_input = widgets.Text(
    value='https://ftp.ebi.ac.uk/pub/databases/metagenomics/gsoc_ssn/networks/new_networks/MGYP001208449037/',
    placeholder='Enter FTP link to the cluster directory',
    description='FTP link:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='700px')
)

display(cluster_rep_input)

Type in the FTP link to the cluster directory you want to visualize, then run the next cell to fetch the data.

In [ ]:
raw_url = cluster_rep_input.value.strip()

CLUSTER_REP = raw_url.strip('/').split('/')[-1]

base_url = raw_url if raw_url.endswith('/') else f"{raw_url}/"

nodes_url = f"{base_url}{CLUSTER_REP}_nodes.parquet"
edges_url = f"{base_url}{CLUSTER_REP}_edges.parquet"

print(f"Fetching data for Cluster: {CLUSTER_REP}...")

nodes_df = duckdb.sql(f"SELECT * FROM '{nodes_url}'").df()
edges_df = duckdb.sql(f"SELECT * FROM '{edges_url}'").df()

print("Data loaded successfully!")
display(nodes_df.head())
display(edges_df.head())

### Option B: Read files locally

Select the `<cluster_rep>_nodes.parquet` and `<cluster_rep>_edges.parquet` files generated by the MiSSN pipeline or the `build_ssn.py` script. They can be found under the `/networks/<cluster_rep>` subdirectory.

> **Note:** If you are using custom files that were **not** generated by the MiSSN pipeline, please ensure they are `.parquet` files with the correct column headers (`id`, `biome`, and `color` for the nodes file; `source`, `target`, and `sequence_identity` for the edges file).

> **Important:** Only use Option A OR Option B. If you load data here, it will overwrite any data fetched via FTP.

In [ ]:
# Create separate filechoosers for the nodes and edges 
fc_nodes = FileChooser('..')
fc_nodes.title = '<b>Select your _nodes.parquet file</b>'
fc_nodes.filter_pattern = '*_nodes.parquet' 

fc_edges = FileChooser('..')
fc_edges.title = '<b>Select your _edges.parquet file</b>'
fc_edges.filter_pattern = '*_edges.parquet' 

load_button = widgets.Button(description="Load local data", button_style='primary')
output = widgets.Output()

# Define what happens when the button is clicked
def load_local_data(b):
    with output:
        clear_output()
        
        # Ensure both files are selected
        if not fc_nodes.selected or not fc_edges.selected:
            print("Please select BOTH the nodes and edges files from the pickers above!")
            return
            
        print("Loading local Parquet files into memory...")
        
        # Use 'global' so df and all_cluster_sizes are saved into the notebook's 
        # memory and can be used in other cells later
        global nodes_df, edges_df, CLUSTER_REP
        
        CLUSTER_REP = fc_nodes.selected_filename.split('_')[0]
        
        nodes_df = pd.read_parquet(fc_nodes.selected)
        edges_df = pd.read_parquet(fc_edges.selected)

        print("Data loaded successfully!")
        
        # Display the table previews
        display(widgets.HTML("<h3>Nodes</h3>"))
        display(nodes_df.head(5))
        
        display(widgets.HTML("<h3>Edges</h3>"))
        display(edges_df.head(5))

# Attach the click event to the button
load_button.on_click(load_local_data)

ui_panel = widgets.VBox([fc_nodes, fc_edges, load_button, output])
display(ui_panel)

## Build the interactive dashboard

Initialize the Cosmograph template, custom styling functions, and filtering widgets.

### Function to set the cluster representative's size and shape different from the other points.

In [ ]:
def set_cluster_rep_style(kwargs):
    # Make sure 'points' exist in the kwargs
    # so the ingress function does operations
    # only when point data is fully loaded
    if "points" in kwargs and kwargs["points"] is not None:
        points_df = kwargs["points"]
        
        # Add the size and shape column to the temporary copy
        points_df['size'] = 5.0
        points_df.loc[points_df['id'] == CLUSTER_REP, 'size'] = 20.0

        # Tell the widget to use this new temporary column
        kwargs["point_size_by"] = "size"
        
    return kwargs

### Create the Cosmograph blueprint

We create a reusable template, determining the style.
All configuration parameters can be found here [here](https://cosmograph.app/docs-widget/configuration/).

In [ ]:
graph_style = cosmo(
    # Obligatory
    point_id_by="id",
    link_source_by="source",
    link_target_by="target",
    
    # Uses function in the cell above to set cluster rep style
    ingress=set_cluster_rep_style,
    
    # Additional points configuration
    point_color_by="color",
    point_color_strategy="direct",
    point_size_strategy="direct",
    point_label_by="id",
    point_include_columns=["biome"],
    
    # Additional links configuration
    link_strength_by="sequence_identity",
    
    # Graph appearance
    point_greyout_opacity=0.2, 
    render_hovered_point_ring=True,
    link_greyout_opacity=0.2,
    enable_drag=True,
    fit_view_on_init=False,
    
    # Additional parameters
    select_point_on_click=True,
    reset_selection_on_empty_canvas_click=True,
    
    # Labels
    show_labels_for=[CLUSTER_REP],
    
    # Legends
    disable_point_color_legend=True,
    disable_point_size_legend=True,
)

### Add search bars

In [ ]:
search_name = widgets.Text(
    description='Search by ID',
    layout=widgets.Layout(width='300px', margin='0 10px 0 0'),
    style={'description_width': 'initial'}
)

search_biome = widgets.Text(
    description='Search by biome', 
    layout=widgets.Layout(width='300px'),
    style={'description_width': 'initial'}
)

### Add filter

In [ ]:
# Create the color map directly from the unique biomes and their assigned color column in nodes_df
biome_color_map = nodes_df.drop_duplicates(subset=['biome']).set_index('biome')['color'].to_dict()

# Alphabetical sorting for the UI list, pushing the combined ones to the end
ui_display_order = sorted(
    nodes_df['biome'].dropna().unique().tolist(), 
    key=lambda x: (';' in x, x)
)

biome_toggles = []
css_rules = []

# Load the external CSS file
with open('notebook_resources/biome_buttons.css', 'r') as f:
    css_rules.append(f.read())

def on_biome_toggle(change):
    if change['new']:
        change.owner.add_class('is-active')
    else:
        change.owner.remove_class('is-active')

for i, biome in enumerate(ui_display_order):
    color = biome_color_map.get(biome, '#CCCCCC')
    display_text = ';'.join([part.strip() for part in biome.split(';')])
    
    tb = widgets.ToggleButton(
        value=True,
        description=display_text, 
        tooltip='', 
        layout=widgets.Layout(width='99%', flex='0 0 auto')
    )
    
    custom_class = f'biome-color-{i}'
    
    tb.add_class('biome-btn')
    tb.add_class('is-active')
    tb.add_class(custom_class)
    
    css_rules.append(f'''
    .biome-btn.{custom_class} {{
        border-left: 16px solid {color} !important;
    }}
    ''')
    
    tb.observe(on_biome_toggle, names='value')
    
    biome_toggles.append((biome, tb))

css_widget = widgets.HTML("<style>\n" + "\n".join(css_rules) + "\n</style>")

# Scrollable container
biome_filter_box = widgets.VBox(
    [css_widget] + [tb for _, tb in biome_toggles],
    layout=widgets.Layout(
        height='260px',
        overflow_y='auto', 
        border='none',
        padding='8px', 
        width='100%',
        background_color='#2D313A'
    )
)

biome_filter_box.add_class('hide-x-scroll')

None # Not to display filter widget here

# Assembling the dashboard and calling the widget object

This final cell wires up the search bars and biome filters, assembles the UI components, and displays the complete interactive dashboard.

A few things to note:
- The graph is rendered locally using your GPU.
- Typing in the search boxes or clicking the biome toggles will dynamically update the graph and auto-zoom to your selections.
- The cluster representative (determined by linclust) is rendered bigger.

In [ ]:
with open('notebook_resources/dashboard.css', 'r') as f:
    dark_theme_css = widgets.HTML(f"<style>\n{f.read()}\n</style>")

search_name.add_class('force-dark-input')
search_biome.add_class('force-dark-input')
biome_filter_box.add_class('force-dark-container')

is_batch_updating = False

btn_select_all = widgets.Button(description="Select All", layout=widgets.Layout(width='100px', margin='0 6px 0 0'))
btn_deselect_all = widgets.Button(description="Deselect All", layout=widgets.Layout(width='100px'))

btn_select_all.add_class('dark-action-btn')
btn_deselect_all.add_class('dark-action-btn')

def on_select_all_clicked(b):
    global is_batch_updating
    is_batch_updating = True
    for _, tb in biome_toggles:
        tb.value = True
    is_batch_updating = False
    update_graph_selection() 

def on_deselect_all_clicked(b):
    global is_batch_updating
    is_batch_updating = True
    for _, tb in biome_toggles:
        tb.value = False
    is_batch_updating = False
    update_graph_selection() 

btn_select_all.on_click(on_select_all_clicked)
btn_deselect_all.on_click(on_deselect_all_clicked)

filter_label = widgets.HTML('<span style="color: #ffffff; margin-right: 6px;">Filter by biome</span>')
button_group = widgets.HBox(
    [filter_label, btn_select_all, btn_deselect_all], 
    layout=widgets.Layout(margin='0 0 10px 0', align_items='center')
)

def update_graph_selection(change=None):
    if is_batch_updating:
        return 
            
    selected_biomes = [b for b, tb in biome_toggles if tb.value]
    
    if not selected_biomes:
        graph.select_points_by_ids([])  # Pass an empty list to clear the selection
        graph.fit_view()
        return
    
    matches = nodes_df[nodes_df['biome'].isin(selected_biomes)]
    
    name_val = search_name.value.strip().lower()
    if name_val:
        matches = matches[matches['id'].str.lower().str.contains(name_val, na=False, regex=False)]
        
    biome_val = search_biome.value.strip().lower()
    if biome_val:
        matches = matches[matches['biome'].str.lower().str.contains(biome_val, na=False, regex=False)]
        
    matching_ids = matches['id'].tolist()
    
    if matching_ids:
        graph.select_points_by_ids(matching_ids)
        if len(matching_ids) < 100:
            graph.fit_view_by_ids(matching_ids, duration=500, padding=0.1)
    else:
        graph.select_points_by_ids([])
        graph.fit_view()

# Attach observers
search_name.observe(update_graph_selection, names='value')
search_biome.observe(update_graph_selection, names='value')
for _, tb in biome_toggles:
    tb.observe(update_graph_selection, names='value')

# Final layout assembly
ui_panel_top = widgets.HBox([search_name, search_biome], layout=widgets.Layout(margin='0px'))
ui_panel_top.add_class('force-dark-container') 

graph = graph_style(points=nodes_df, links=edges_df)

ui_panel_bottom = widgets.VBox([
    button_group,
    biome_filter_box
], layout=widgets.Layout(margin='0px'))
ui_panel_bottom.add_class('force-dark-container') 

dashboard = widgets.VBox(
    [dark_theme_css, ui_panel_top, graph, ui_panel_bottom], 
    layout=widgets.Layout(margin='0px', padding='0px')
)
dashboard.add_class('master-dark-wrapper')

display(dashboard)